# Rotorpy/ardupilot reference frames

This notebook describes the reference frames used in `rotorpy` and Ardupilot. They are summarized in the following table:

| System    | Body Frame                                | World Frame                            |
|-----------|-------------------------------------------|----------------------------------------|
| Rotorpy   | FLU ($x$: Forward, $y$: Left, $z$: Up )   | ENU ($x$: East, $y$: North, $z$: Up)   |
| Ardupilot | FRD ($x$: Forward, $y$: Right, $z$: Down) | NED ($x$: North, $y$: East, $z$: Down) |

![Reference Frames](https://docs.px4.io/v1.12/assets/img/ref_frames.b0d97b5d.png) 

[[image source - px4 docs]](https://docs.px4.io/v1.12/en/ros/external_position_estimation.html#reference-frames-and-ros)

In [12]:
from scipy.spatial.transform import Rotation
import numpy as np

flu2frd = Rotation.from_euler('x', np.pi)

# Set print options to display 3 decimal places
np.set_printoptions(precision=3)
print(flu2frd.as_matrix())



[[ 1.000e+00  0.000e+00  0.000e+00]
 [ 0.000e+00 -1.000e+00 -1.225e-16]
 [ 0.000e+00  1.225e-16 -1.000e+00]]


## Active and passive rotations

An active rotation is one that takes a vector expressed in a reference frame and rotates it to be in a different orientation in the same reference frame.

Let's take for example the vector $v = [1, 0, 0]$. We have a `Rotation` object $R$ representing a rotation of 
$\pi$ radians around the $Z$ axis. We want to understand whether $R$ represents an active rotation or a passive one.

If the rotation is active it means that performing the operation $v'=Rv$ will result in a vector $v'=[0,1,0]$.

If the rotation is passive this means that the reference frame will be rotated $90\deg$ resulting in the new $x$ axis being aligned with the previous $y$ axis and the previous $y$ being aligned with the _negative_ $x$ axis. Thus the vector $v'$ will be the previous vector represented in the new reference frame $v'=[0,-1,0]$.

Let's check with code!


In [13]:
R = Rotation.from_euler('z', np.pi/2)

v = np.array([1,0,0])

v_prime = R.as_matrix()@ v

if np.allclose(v_prime, [0,1,0]):
    print("The object R represents an active rotation")
elif np.allclose(v_prime, [0,-1,0]):
    print("The object R represents a passive rotation")
else:
    print("Something went wrong..")
    

The object R represents an active rotation


So we understood that the rotation object R represents an **active** rotation. We are going to need this information soon!

The physics backend used to simulate quadrotors in the Duckiematrix is [`rotorpy`](https://github.com/spencerfolk/rotorpy)[^rotorpy]. This simulator provides us the _passive_ rotation $R_{FLU \rightarrow NED}$ from the FLU body frame ($x$ pointing forward, $y$ left and $z$ up) to the ENU inertial frame ($x$ pointing east, $y$ north and $z$ up).

This means that if we want to express a vector $v_B$ from the body reference frame into the corresponding inertial reference vector $v_I$ we have to perform the operation $v_I=R_{FLU \rightarrow NED}v_B$. 

Note that this can also be interpreted as the attitude of the drone in the inertial frame.
To show this let's define for clarity:

$$R_{B\rightarrow I} = R_{FLU \rightarrow NED}$$ 

This is the _passive_ rotation from the body frame to the inertial frame. The attitude is defined as the _active_ rotation from the inertial frame to the body frame. This is equivalent to inverting the matrix two times:

$$R_{B\rightarrow I}^{active}=\big(R_{B\rightarrow I}\big)^{-1}$$

To obtain the active rotation matrix from the body frame to the reference frame. And then once more 

$$R_{I\rightarrow B}^{active}= \big(R_{B\rightarrow I}^{active}\big)^{-1}=\bigg[\big(R_{B\rightarrow I}\big)^{-1}\bigg]^{-1}$$

to obtain the _active_ rotation matrix from the inertial frame to the body frame. But since inverting twice a matrix results in the original matrix this means that:

$$\bigg[\big(R_{B\rightarrow I}\big)^{-1}\bigg]^{-1}=R_{B\rightarrow I}=R_{I\rightarrow B}^{active}$$

and hence the attitude.

[^rotorpy]: 
    ```bibtex
    @article{folk2023rotorpy,
        title={{RotorPy}: A Python-based Multirotor Simulator with Aerodynamics for Education and Research},
        author={Folk, Spencer and Paulos, James and Kumar, Vijay},
        journal={arXiv preprint arXiv:2306.04485},
        year={2023}
    }
    ```

## Ardupilot - `rotorpy` rotation conventions

The attitude in Ardupilot is represented as a quaternion in scalar-first representation $$q_{ARDU}=[w, x, y, z]$$ and representing the active rotation from the world frame (NED) to the body frame (FRD). As seen in the previous section, it also represents the passive rotation from the body frame to the world frame.

In rotorpy the attitude representation is a scalar-last quaternion  $$q_{rotorpy}=[x, y, z, w]$$ and it represents the rotation from the body frame (GLU) to the world frame (ENU).

In [15]:
R_flu2enu = Rotation.from_quat([0,0,0,1]) # Attitude of FLU frame in ENU (i.e. the _passive_ transform _from_ FLU _to_ ENU)

M_flu2frd = R.from_euler('x', np.pi) # Active rotation from FLU to FRD
M_enu2ned = R.from_matrix([[0, 1, 0], [1, 0, 0], [0, 0, -1]]) # Active rotation from ENU to NED

# Note how the FLU2FRD rotation is equivalent to its inverse
assert np.allclose(M_flu2frd.inv().as_matrix(),M_flu2frd.as_matrix())

# Note how the ENU2NEd rotation is equivalent to its inverse
assert np.allclose(M_enu2ned.inv().as_matrix(), M_enu2ned.as_matrix())
# This is because their transpose are equivalent and the inverse of a rotation matrix is just its transpose
assert np.allclose(M_enu2ned.as_matrix().T, M_enu2ned.inv().as_matrix())

# 2. Obtain the rotation from the body frame (FRD) to the world frame (NED)
# This is the attitude of the FRD frame in the NED frame
R_frd2ned = M_enu2ned * R_flu2enu * M_flu2frd

M_frd2flu = M_flu2frd # It should actually be M_flu2frd.inv() but we have shown this to equivalent to M_flu2frd
M_enu2ned = M_enu2ned # It should properly be the inverse of this since we defined it as a passive transformation but they are equivalent

# This is the overall passive rotation representing the transformation from the FRD body frame to the NED inertial reference frame
R_frd2ned_check = M_enu2ned * R_flu2enu * M_frd2flu